In [1]:
import requests
import pandas as pd
import geopandas as gpd

## Dataset 1: Complaints

In [ ]:
base_url_complaints = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

params_complaints = {
    "$select": "unique_key,created_date, agency_name, complaint_type, incident_zip, city, council_district,borough",
    "$where": "agency = 'HPD'",
    #"$order": "created_date DESC"
    "$limit": 500000
}

response = requests.get(base_url_complaints, params=params_complaints)
data = response.json()
df_complaints = pd.DataFrame(data)
df_complaints.head(10)

: 

In [ ]:
# df_complaints.isna().sum()
# df_clean = df_complaints.dropna().copy()

: 

In [ ]:
print(f"Number of observations: {len(df_complaints)}")

: 

In [ ]:
print("Diferent complaint types:")
df_complaints["complaint_type"].unique()

: 

In [ ]:
print(f"Number of unique incident zip codes: {len(df_complaints['incident_zip'].unique())}")

: 

In [ ]:
print("Diferent complaint types:")
len(df_complaints["council_district"].unique())

: 

## Dataset 2: Violations

In [ ]:
base_url_violations = "https://data.cityofnewyork.us/resource/wvxf-dwi5.json"

params_violations = {
    "$select": "violationid, zip, class, inspectiondate, originalcorrectbydate, newcorrectbydate, certifieddate, councildistrict",
    "$where": "inspectiondate IS NOT NULL AND originalcorrectbydate IS NOT NULL AND newcorrectbydate IS NOT NULL AND certifieddate IS NOT NULL AND class='C'"  # Filter out NULL values
}

response = requests.get(base_url_violations, params=params_violations)
data = response.json()
df_violations = pd.DataFrame(data)
df_violations.head(10)

: 

In [ ]:
print(f"Number of observations: {len(df_violations)}")

: 

In [ ]:
print(f"Number of unique incident zip codes: {len(df_violations['zip'].unique())}")

: 

## Dataset 3: Income per area

2018-2022 ACS 5-Year Estimates for ZIP 10001:

Survey Year → Question Asked
────────────────────────────
2018 → "Income in past 12 months (2017-2018)?" → 500 households answer
2019 → "Income in past 12 months (2018-2019)?" → 500 households answer  
2020 → "Income in past 12 months (2019-2020)?" → 500 households answer
2021 → "Income in past 12 months (2020-2021)?" → 500 households answer
2022 → "Income in past 12 months (2021-2022)?" → 500 households answer

Census pools all 2,500 responses → calculates estimate for ZIP 10001

1. Find Available 5 Years Subject Tables

In [ ]:
groups_url = "https://api.census.gov/data/2022/acs/acs5/subject/groups.json"
response = requests.get(groups_url)
groups = response.json()
groups_df = pd.DataFrame(groups['groups'])
groups_df.head(20)

: 

2. Income related tables have S19

In [ ]:
income_tables = groups_df[groups_df['name'].str.contains('S19', na=False)]
print(income_tables[['name', 'description']])

: 

3. Getting all  variables in a S19 table, use mean table

S1901_C01_012E
  │    │  │  └─ E = Estimate, M = Margin of Error
  │    │  └──── Variable number (001, 002, 003...)
  │    └─────── Column: C01 = Total, C02 = Male, C03 = Female
  └──────────── Table S1901

In [ ]:
table_url = "https://api.census.gov/data/2024/acs/acs5/subject/groups/S1901.json"
response = requests.get(table_url)
table_info = response.json()

variables = []
for var_name, var_info in table_info['variables'].items():
    if var_name != 'for' and var_name != 'in':  # Skip geography variables
        variables.append({
            'variable': var_name,
            'label': var_info.get('label', ''),
            'concept': var_info.get('concept', '')
        })

vars_df = pd.DataFrame(variables)
print(f"Total variables in S1901: {len(vars_df)}")
vars_df.head(10)

: 

In [ ]:
pd.set_option('display.max_colwidth', None)  # Show full text
vars_df[['variable', 'label']].head(10)

: 

This is the variable we want, it is median income.

In [ ]:
vars_df[vars_df["variable"] == "S1901_C01_012E"]

: 

4. Getting median income for all zip codes in new york

In [ ]:
base_url = "https://api.census.gov/data/2024/acs/acs5/subject"

params = {
    "get": "NAME,S1901_C01_012E,S1901_C01_012M", #this is median
    "for": "zip code tabulation area:*"
}

print("Fetching data from Census API...")
response = requests.get(base_url, params=params)

if response.status_code == 200:
    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ['location', 'median_income', 'margin_of_error', 'zipcode']
    
    df['median_income'] = pd.to_numeric(df['median_income'], errors='coerce')
    df['margin_of_error'] = pd.to_numeric(df['margin_of_error'], errors='coerce')
    
    # filter for new york city, not new york state(5 boroughs)
    def is_nyc_zip(zipcode):
        if zipcode.startswith('100') or zipcode.startswith('101') or zipcode.startswith('102'):
            return True  # Manhattan
        elif zipcode.startswith('103'):
            return True  # Staten Island
        elif zipcode.startswith('104'):
            return True  # Bronx
        elif zipcode.startswith('110') or zipcode.startswith('111') or zipcode.startswith('113') or zipcode.startswith('114') or zipcode.startswith('116'):
            return True  # Queens
        elif zipcode.startswith('112'):
            return True  # Brooklyn
        else:
            return False
    
    # Filter for nyc
    nyc_df = df[df['zipcode'].apply(is_nyc_zip)].copy()

    # Replace Census missing-value codes with NaN
    missing_codes = [-666666666]
    nyc_df['median_income'] = nyc_df['median_income'].replace(missing_codes, pd.NA)
    nyc_df['margin_of_error'] = nyc_df['margin_of_error'].replace(missing_codes, pd.NA)
    nyc_df = nyc_df[nyc_df['median_income'].notna()]
    nyc_df = nyc_df.sort_values('median_income', ascending=False).reset_index(drop=True)
    
    # Add borough column
    def get_borough(zipcode):
        if zipcode.startswith('100') or zipcode.startswith('101') or zipcode.startswith('102'):
            return 'Manhattan'
        elif zipcode.startswith('103'):
            return 'Staten Island'
        elif zipcode.startswith('104'):
            return 'Bronx'
        elif zipcode.startswith('110') or zipcode.startswith('111') or zipcode.startswith('113') or zipcode.startswith('114') or zipcode.startswith('116'):
            return 'Queens'
        elif zipcode.startswith('112'):
            return 'Brooklyn'
        else:
            return 'Unknown'
    
    nyc_df['borough'] = nyc_df['zipcode'].apply(get_borough)
    
    print(f"Total NYC ZIP codes: {len(nyc_df)}")

    nyc_df.to_csv('nyc_zipcode_income.csv', index=False)
    print(f"\n Saved to 'nyc_zipcode_income.csv'")
    
else:
    print(f"Error {response.status_code}: {response.text}")

: 

In [ ]:
# Borough statistics
print(f"ZIP Codes by Borough")
print(nyc_df['borough'].value_counts().sort_index())

print(f"\nTop 10 highest income ZIP codes in NYC:")
print(nyc_df[['zipcode', 'borough', 'median_income']].head(10))

print(f"\nBottom 10 lowest income ZIP codes in NYC:")
print(nyc_df[['zipcode', 'borough', 'median_income']].tail(10))

print(f"\nMedian Income by Borough")
borough_stats = nyc_df.groupby('borough')['median_income'].agg(['mean', 'median', 'min', 'max', 'count'])
borough_stats.columns = ['Average', 'Median', 'Lowest', 'Highest', 'ZIP_Count']
print(borough_stats.round(0))


: 

In [ ]:
# NYC statistics
print(f"\nOverall NYC Income Statistics")
print(f"Highest: ${nyc_df['median_income'].max():,.0f}")
print(f"Lowest: ${nyc_df['median_income'].min():,.0f}")
print(f"Average: ${nyc_df['median_income'].mean():,.0f}")
print(f"Median: ${nyc_df['median_income'].median():,.0f}")

: 

## Council District GeoJSON

In [ ]:
cd_g = gpd.read_file(r"C:\Users\Diogo Forte\OneDrive\DTU\4TH SEMESTER\Social data analysis\final_project\NYC City Council Districts.geojson")

print(cd_g.shape)

: 

In [ ]:
# Get bounds directly from the GeoDataFrame geometry
lon_min, lat_min, lon_max, lat_max = cd_g.total_bounds

lon_center = (lon_min + lon_max) / 2
lat_center = (lat_min + lat_max) / 2

print(f"Longitude: {lon_min:.4f} to {lon_max:.4f}, center: {lon_center:.4f}")
print(f"Latitude:  {lat_min:.4f} to {lat_max:.4f}, center: {lat_center:.4f}")

: 

In [ ]:
import plotly.express as px

fig = px.choropleth_map(
    cd_g,
    geojson=cd_g,
    locations='coun_dist',
    featureidkey="properties.coun_dist",
    color='shape_area',
    color_continuous_scale="Viridis",
    map_style="carto-positron",
    zoom=9,
    center={"lat": 40.71, "lon": -74.00},
    opacity=0.5,
    labels={"shape_area": "Area"}
)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()


: 

## Zip Code GeoJSON

In [ ]:
zp_g = gpd.read_file(r"C:\Users\Diogo Forte\OneDrive\DTU\4TH SEMESTER\Social data analysis\final_project\nyc-zip-code-tabulation-areas-polygons.geojson")

print(zp_g.shape)

: 

In [ ]:
# Get bounds directly from the GeoDataFrame geometry
lon_min, lat_min, lon_max, lat_max = zp_g.total_bounds

lon_center = (lon_min + lon_max) / 2
lat_center = (lat_min + lat_max) / 2

print(f"Longitude: {lon_min:.4f} to {lon_max:.4f}, center: {lon_center:.4f}")
print(f"Latitude:  {lat_min:.4f} to {lat_max:.4f}, center: {lat_center:.4f}")


: 

In [ ]:
fig = px.choropleth_map(
    zp_g,
    geojson=zp_g,
    locations='postalCode',
    featureidkey="properties.postalCode",
    color='Shape_Area',
    color_continuous_scale="Viridis",
    map_style="carto-positron",
    zoom=9,
    center={"lat": 40.71, "lon": -74.00},
    opacity=0.5,
    labels={"Shape_Area": "Area"}
)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

: 

## Map ZIP Codes to NYC City Council Districts

In [ ]:
# Reproject both layers to NY State Plane (EPSG:2263) for accurate area calculations
cd_proj = cd_g.to_crs(epsg=2263)[['coun_dist', 'geometry']]
zp_proj = zp_g.to_crs(epsg=2263)[['postalCode', 'geometry']]

# Compute polygon intersections between every ZIP code and every council district
intersection = gpd.overlay(zp_proj, cd_proj, how='intersection')
intersection['area'] = intersection.geometry.area

# Primary council district = the one with the largest intersection area per ZIP code
primary_district = (
    intersection.sort_values('area', ascending=False)
    .groupby('postalCode', as_index=False)
    .first()[['postalCode', 'coun_dist']]
    .rename(columns={'coun_dist': 'council_district'})
)
primary_district['council_district'] = pd.to_numeric(primary_district['council_district'], errors='coerce').astype('Int64')

# Ensure zipcode is string for merge
nyc_df['zipcode'] = nyc_df['zipcode'].astype(str)
primary_district['postalCode'] = primary_district['postalCode'].astype(str)

# Merge both mappings into the income DataFrame
nyc_df_cd = nyc_df.merge(primary_district, left_on='zipcode', right_on='postalCode', how='left').drop(columns='postalCode')

# Save updated CSV
nyc_df_cd.to_csv('nyc_zipcode_income.csv', index=False)
print(f'Updated CSV saved: {len(nyc_df_cd)} rows')
print(f'ZIP codes with a council district mapped: {nyc_df_cd["council_district"].notna().sum()}')
nyc_df_cd[['zipcode', 'median_income', 'borough', 'council_district']].head(10)


: 

In [ ]:
# All <NA> council districts
nyc_df_cd = nyc_df_cd[nyc_df_cd['council_district'].isna()]
print(f"ZIP codes with no council district mapped: {len(nyc_df_cd)}")
print(nyc_df_cd[['zipcode', 'borough']].head(9))

: 

## Why does 11249 not exist in the GeoJSON?

11249 is a relatively new ZIP code in Williamsburg/North Brooklyn that was introduced by USPS specifically for a large residential development (primarily 1 North 4th Place / Northside Piers). It's a non-standard ZCTA (ZIP Code Tabulation Area) — the US Census Bureau, which defines ZCTAs for mapping purposes, did not include it as a separate tabulation area. The GeoJSON you're using is based on NYC's ZCTA boundaries (Census-derived), which predate or exclude this delivery-only ZIP code.

In short: your dataset has records using USPS ZIP code 11249, but your GeoJSON uses Census ZCTAs, and 11249 was never assigned a ZCTA polygon. You'll see the same issue for the other unmapped Queens ZIPs — they are either outside NYC proper (like Great Neck/Nassau County addresses: 11020, 11021, 11023, 11024, 11030, 11050) or similar edge cases.